# tempify v0.1.6 — API ergonómica tipo `terra` de R

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/djwillichile/tempify/blob/main/docs/tutorials/03-ergonomic-api.ipynb)

**Cargar, inspeccionar, visualizar e interpolar un stack en pocas líneas.**

Desde la v0.1.6, `tempify` expone tres símbolos en el nivel raíz del paquete:

```python
from tempify import rast, tempify, plot
```

Estos tres símbolos están inspirados directamente en el paquete `terra` de R y
permiten un flujo exploratorio muy compacto. Para flujos de producción con
validación completa y escritura multi-formato, el `TempifyPipeline` clásico
sigue disponible (ver el notebook `01-getting-started`).

## 1. Instalación

En Google Colab la primera ejecución toma ~1 min (instala `rasterio`/`gdal`).

In [ ]:
try:
    import tempify
    assert tempify.__version__ >= "0.1.6", "Requiere tempify >= 0.1.6"
    print(f"tempify {tempify.__version__} ya instalado.")
except (ImportError, AssertionError):
    from IPython import get_ipython
    get_ipython().run_line_magic(
        "pip",
        "install --upgrade git+https://github.com/djwillichile/tempify.git",
    )
    import importlib, tempify
    importlib.reload(tempify)
    print(f"tempify {tempify.__version__} instalado.")

## 2. Preparamos un GeoTIFF multi-banda de ejemplo

Para que el notebook sea autocontenido, generamos un TIF de 12 bandas con
ciclo anual sintético (temperatura media mensual tipo WorldClim) sobre una
grilla 30×30 px en EPSG:4326. Si tienes tu propio archivo, salta esta celda
y apunta `MI_TIF` a tu ruta.

In [ ]:
from pathlib import Path
import numpy as np
import rioxarray  # noqa: F401
import xarray as xr

rng = np.random.default_rng(42)
months = np.arange(12).reshape(12, 1, 1)
lat_gradient = np.linspace(0, -4, 30).reshape(1, 30, 1)
alt_gradient = np.linspace(0, -6, 30).reshape(1, 1, 30)
annual_cycle = 15.0 + 10.0 * np.sin(2 * np.pi * (months - 1) / 12 - np.pi / 2)
arr = (annual_cycle + lat_gradient + alt_gradient + rng.normal(0, 0.3, (12, 30, 30))).astype(np.float32)

da = xr.DataArray(
    arr,
    dims=("band", "y", "x"),
    coords={
        "band": list(range(1, 13)),
        "y": np.linspace(-32.5, -34.5, 30),
        "x": np.linspace(-71.5, -69.5, 30),
    },
    name="tavg_monthly",
).rio.write_crs("EPSG:4326")

MI_TIF = Path("worldclim_tavg_12bandas.tif")
da.rio.to_raster(MI_TIF)
print(f"GeoTIFF multi-banda creado en: {MI_TIF}  ({MI_TIF.stat().st_size / 1024:.1f} KB)")

## 3. Cargar e inspeccionar — `rast()` + `print()` + `.str()`

Una sola línea para cargar, y `print(r)` muestra info compacta tipo `terra::print(r)`.

In [ ]:
from tempify import rast, tempify, plot

r = rast(MI_TIF)
print(r)

El método `.str()` añade información adicional: rango de valores, NaN, atributos.

In [ ]:
r.str()

## 4. Visualización — `plot()`

`plot(r)` muestra una grilla automática con colorbar compartida. El layout se
calcula como `ceil(sqrt(n)) × ceil(n / ncols)`.

In [ ]:
plot(r)

## 5. Densificación temporal — `tempify()`

`tempify(stack, from_freq, to_freq, method)` aplica la interpolación **en memoria**
(sin escribir a disco). El método por defecto es `pchip_mp` (PCHIP con conservación
estricta de media mensual), pero se aceptan los seis: `linear`, `cubic`, `pchip`,
`pchip_mp`, `fourier`, `akima`.

In [ ]:
r2 = tempify(r, from_freq="monthly", to_freq="daily", method="cubic", year=2023)
print(r2)

In [ ]:
r2.str()

## 6. Visualizar un subset de bandas — `plot(r2, sub=...)`

El argumento `sub` es **1-indexado** (como en `terra::plot(r, sub=1:16)`).
Mostramos las primeras 16 bandas diarias (los primeros 16 días del año).

In [ ]:
plot(r2, sub=range(1, 17))

También puedes pasar una lista arbitraria de índices: por ejemplo, los 12 días 15
(uno por mes) para visualizar la climatología reconstruida.

In [ ]:
import datetime as dt
doy15 = [dt.date(2023, m, 15).timetuple().tm_yday for m in range(1, 13)]
plot(r2, sub=doy15)

## 7. El flujo completo en seis líneas

Resumiendo todo lo anterior:

In [ ]:
from tempify import rast, tempify, plot

r  = rast(MI_TIF)
print(r); r.str(); plot(r)

r2 = tempify(r, from_freq="monthly", to_freq="daily", method="cubic", year=2023)
print(r2); r2.str(); plot(r2, sub=range(1, 17))

## 8. Acceder al `xr.DataArray` subyacente

`TempifyRast` es un wrapper liviano. El array de xarray subyacente está disponible
vía la propiedad `.data` (sin copia), por si necesitas operaciones nativas de xarray
o pasarlo a otro paquete (numpy, dask, salem, etc).

In [ ]:
da_diario = r2.data
print(type(da_diario))
print("dims:", da_diario.dims)
print("shape:", da_diario.shape)
print("CRS:", r2.crs)

# Media climatológica del año reconstruido
media_anual = da_diario.mean("time")
print("\nMedia anual reconstruida (4×4 centro):")
print(media_anual.values[13:17, 13:17].round(2))

## 9. Cuándo usar la API ergonómica vs. el pipeline completo

| Caso | Usa esto |
|---|---|
| Exploración rápida, visualización, prototipo | `rast()` + `tempify()` + `plot()` (v0.1.6) |
| Producción: validación geoespacial pre/post, reporte de procedencia, escritura NetCDF/Zarr | `TempifyPipeline` + `PipelineConfig` |

La capa ergonómica **no** valida CRS/resolución, **no** verifica conservación de media
post-interpolación, **no** escribe a disco, y **no** genera reporte. Si necesitas todo
eso, usa el pipeline completo (ver notebook `01-getting-started`).

Ambas capas son compatibles: el `xr.DataArray` que retorna `r2.data` puede pasarse
directamente a otras herramientas o serializarse con xarray/rioxarray manualmente.

---

**Versión cubierta:** tempify 0.1.6 · **API estable:** `rast`, `tempify`, `plot`, `TempifyRast`

**Issues / sugerencias:** [github.com/djwillichile/tempify/issues](https://github.com/djwillichile/tempify/issues)